# Part III: Computation - Code Examples

This notebook covers Chapters 9-12:
- **Chapter 9**: Quantum Gates — unitary transformations
- **Chapter 10**: Interference as Computation — the core trick
- **Chapter 11**: Quantum Algorithms — what they do
- **Chapter 12**: Why Quantum Speedup? — when it helps

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

## Chapter 9: Quantum Gates

Gates are unitary matrices that transform quantum states.

In [ ]:
# Standard gates
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)  # NOT
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)  # Phase flip
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)  # Hadamard
S = np.array([[1, 0], [0, 1j]], dtype=complex)  # π/2 phase
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)  # π/4 phase

# CNOT gate
CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], dtype=complex)


def tensor_product(A, B):
    return np.kron(A, B)


print("=== Standard Quantum Gates ===")
print(f"\nX (NOT):\n{X.real.astype(int)}")
print(f"\nZ (Phase flip):\n{Z.real.astype(int)}")
print(f"\nH (Hadamard):\n{H.real}")

=== Standard Quantum Gates ===

X (NOT):
[[0 1]
 [1 0]]

Z (Phase flip):
[[ 1  0]
 [ 0 -1]]

H (Hadamard):
[[ 0.707  0.707]
 [ 0.707 -0.707]]


In [ ]:
# Verify unitarity
def is_unitary(U):
    return np.allclose(U @ U.conj().T, np.eye(len(U)))


print("Unitarity check:")
for name, gate in [("X", X), ("H", H), ("CNOT", CNOT)]:
    print(f"  {name}: U†U = I? {is_unitary(gate)}")

Unitarity check:
  X: U†U = I? True
  H: U†U = I? True
  CNOT: U†U = I? True


In [4]:
# Bell state creation: H on qubit 1, then CNOT
ket_0 = np.array([1, 0], dtype=complex)
ket_00 = tensor_product(ket_0, ket_0)

print("=== Bell State Creation ===")
print(f"Initial: |00⟩ = {ket_00}")

# H ⊗ I
state = tensor_product(H, I) @ ket_00
print(f"After H⊗I: {state}")

# CNOT
state = CNOT @ state
print(f"After CNOT: {state}")
print("\n→ This is |Φ+⟩ = (|00⟩ + |11⟩)/√2!")

=== Bell State Creation ===
Initial: |00⟩ = [1.+0.j 0.+0.j 0.+0.j 0.+0.j]
After H⊗I: [0.707+0.j 0.   +0.j 0.707+0.j 0.   +0.j]
After CNOT: [0.707+0.j 0.   +0.j 0.   +0.j 0.707+0.j]

→ This is |Φ+⟩ = (|00⟩ + |11⟩)/√2!


## Chapter 10: Interference

The core trick: wrong answers cancel, right answers remain.

In [ ]:
def hadamard_n(n):
    """n-qubit Hadamard gate."""
    result = H
    for _ in range(n - 1):
        result = tensor_product(result, H)
    return result


def oracle_constant(n):
    """Oracle for constant function: f(x) = 0 for all x."""
    return np.eye(2**n, dtype=complex)


def oracle_balanced(n):
    """Oracle for balanced function: f(x) = x mod 2."""
    dim = 2**n
    O = np.eye(dim, dtype=complex)
    for x in range(dim):
        if x % 2 == 1:  # f(x) = 1 for odd x
            O[x, x] = -1
    return O


def deutsch_jozsa(oracle, n):
    """Deutsch-Jozsa algorithm: distinguish constant from balanced."""
    dim = 2**n
    state = np.zeros(dim, dtype=complex)
    state[0] = 1  # |0...0⟩

    H_n = hadamard_n(n)
    state = H_n @ state  # Superposition
    state = oracle @ state  # Oracle
    state = H_n @ state  # Interference

    prob_zero = np.abs(state[0]) ** 2
    return prob_zero


print("=== Deutsch-Jozsa Algorithm ===")
print("One query distinguishes constant from balanced!\n")

for n in [2, 3, 4]:
    p_const = deutsch_jozsa(oracle_constant(n), n)
    p_bal = deutsch_jozsa(oracle_balanced(n), n)
    print(f"n={n} qubits:")
    print(f"  Constant f: P(|0...0⟩) = {p_const:.3f} → CONSTANT")
    print(f"  Balanced f: P(|0...0⟩) = {p_bal:.3f} → BALANCED")

=== Deutsch-Jozsa Algorithm ===
One query distinguishes constant from balanced!

n=2 qubits:
  Constant f: P(|0...0⟩) = 1.000 → CONSTANT
  Balanced f: P(|0...0⟩) = 0.000 → BALANCED
n=3 qubits:
  Constant f: P(|0...0⟩) = 1.000 → CONSTANT
  Balanced f: P(|0...0⟩) = 0.000 → BALANCED
n=4 qubits:
  Constant f: P(|0...0⟩) = 1.000 → CONSTANT
  Balanced f: P(|0...0⟩) = 0.000 → BALANCED


In [6]:
# Show the interference pattern
print("=== Interference Visualization ===")
print("\nFor 2-qubit Deutsch-Jozsa:")

n = 2
state = np.zeros(2**n, dtype=complex)
state[0] = 1

print(f"Initial:        {state}")

H_n = hadamard_n(n)
state = H_n @ state
print(f"After H⊗H:      {state.real} (uniform superposition)")

# Balanced oracle flips phase on odd indices
state_bal = oracle_balanced(n) @ state
print(f"After oracle:   {state_bal.real} (phases flipped on odd)")

state_final = H_n @ state_bal
print(f"After final H:  {state_final.real}")
print("\n→ Amplitude on |00⟩ is zero! Destructive interference.")

=== Interference Visualization ===

For 2-qubit Deutsch-Jozsa:
Initial:        [1.+0.j 0.+0.j 0.+0.j 0.+0.j]
After H⊗H:      [0.5 0.5 0.5 0.5] (uniform superposition)
After oracle:   [ 0.5 -0.5  0.5 -0.5] (phases flipped on odd)
After final H:  [0. 1. 0. 0.]

→ Amplitude on |00⟩ is zero! Destructive interference.


## Chapter 11: Algorithms

Grover's search: amplitude amplification.

In [ ]:
def grover_oracle(n, marked):
    """Oracle that flips phase of marked state."""
    O = np.eye(2**n, dtype=complex)
    O[marked, marked] = -1
    return O


def grover_diffusion(n):
    """Diffusion operator: 2|s⟩⟨s| - I where |s⟩ is uniform superposition."""
    dim = 2**n
    s = np.ones(dim) / np.sqrt(dim)
    return 2 * np.outer(s, s) - np.eye(dim)


def grover_search(n, marked, iterations):
    """Run Grover's algorithm."""
    dim = 2**n

    # Start with uniform superposition
    state = np.ones(dim, dtype=complex) / np.sqrt(dim)

    oracle = grover_oracle(n, marked)
    diffusion = grover_diffusion(n)

    probs = [np.abs(state[marked]) ** 2]

    for _ in range(iterations):
        state = oracle @ state
        state = diffusion @ state
        probs.append(np.abs(state[marked]) ** 2)

    return probs


print("=== Grover's Search ===")
n = 4  # 16 items
marked = 7  # Looking for item 7
optimal_iterations = int(np.round(np.pi / 4 * np.sqrt(2**n)))

print(f"Searching {2**n} items for item {marked}")
print(f"Optimal iterations: ≈ π√N/4 = {optimal_iterations}")

probs = grover_search(n, marked, optimal_iterations + 2)
print(f"\n{'Iteration':<12} {'P(marked)':<12}")
print("-" * 25)
for i, p in enumerate(probs):
    marker = " ← optimal" if i == optimal_iterations else ""
    print(f"{i:<12} {p:<12.4f}{marker}")

=== Grover's Search ===
Searching 16 items for item 7
Optimal iterations: ≈ π√N/4 = 3

Iteration    P(marked)   
-------------------------
0            0.0625      
1            0.4727      
2            0.9084      
3            0.9613       ← optimal
4            0.5817      
5            0.1255      


## Chapter 12: Why Speedup?

The unified view: off-diagonals are the quantum resource.

In [ ]:
def von_neumann_entropy(rho):
    eigenvalues = np.linalg.eigvalsh(rho)
    eigenvalues = eigenvalues[eigenvalues > 1e-10]
    if len(eigenvalues) == 0:
        return 0.0
    return -np.sum(eigenvalues * np.log2(eigenvalues))


print("=== The Unified View ===")
print("\nDiagonal = classical. Off-diagonal = quantum.\n")

# Same diagonal, different off-diagonals
rho_classical = np.array([[0.5, 0.0], [0.0, 0.5]])
rho_quantum = np.array([[0.5, 0.5], [0.5, 0.5]])

print("Classical (diagonal):")
print(f"  ρ = {rho_classical[0]}")
print(f"      {rho_classical[1]}")
print(f"  Entropy: {von_neumann_entropy(rho_classical):.3f} bits")

print("\nQuantum (off-diagonal):")
print(f"  ρ = {rho_quantum[0]}")
print(f"      {rho_quantum[1]}")
print(f"  Entropy: {von_neumann_entropy(rho_quantum):.3f} bits")

print("\n→ Off-diagonals reduce entropy (compress information).")
print("→ This is the source of quantum advantage.")

=== The Unified View ===

Diagonal = classical. Off-diagonal = quantum.

Classical (diagonal):
  ρ = [0.5 0. ]
      [0.  0.5]
  Entropy: 1.000 bits

Quantum (off-diagonal):
  ρ = [0.5 0.5]
      [0.5 0.5]
  Entropy: -0.000 bits

→ Off-diagonals reduce entropy (compress information).
→ This is the source of quantum advantage.


In [ ]:
# Connection to computational mechanics
print("=== Computational Mechanics Connection ===")
print("\nPerturbed coin: signal state overlap enables C_q < C_μ\n")

for p in [0.5, 0.3, 0.1, 0.01]:
    s0 = np.array([np.sqrt(1 - p), np.sqrt(p)])
    s1 = np.array([np.sqrt(p), np.sqrt(1 - p)])

    overlap = np.abs(np.dot(s0, s1))
    rho_q = 0.5 * np.outer(s0, s0) + 0.5 * np.outer(s1, s1)
    C_q = von_neumann_entropy(rho_q)
    C_mu = 1.0

    print(f"p={p:.2f}: overlap={overlap:.3f}, C_q={C_q:.3f}, advantage={C_mu - C_q:.3f}")

print("\nMore overlap → more off-diagonal → more compression → more advantage!")

=== Computational Mechanics Connection ===

Perturbed coin: signal state overlap enables C_q < C_μ

p=0.50: overlap=1.000, C_q=-0.000, advantage=1.000
p=0.30: overlap=0.917, C_q=0.250, advantage=0.750
p=0.10: overlap=0.600, C_q=0.722, advantage=0.278
p=0.01: overlap=0.199, C_q=0.971, advantage=0.029

More overlap → more off-diagonal → more compression → more advantage!


## Summary

The deep dive is complete. The key insight:

> **Quantum advantage = structured interference using off-diagonal coherence.**

Whether in algorithms (Grover, Shor) or in complexity measures ($C_q < C_\mu$), the mechanism is the same:
- Superposition creates possibility
- Interference selects the answer
- Decoherence destroys the advantage

When you see **diagonal = classical, off-diagonal = quantum**, you understand quantum mechanics.